# Data Check (New Schema)

Проверка `data/index/index.parquet` под новую схему: `condition_*`, `description`, `object_type`, `collected_at`.


In [1]:
from pathlib import Path
import pandas as pd

PARQUET_PATH = Path('/Users/zhasik/Desktop/krisha/data/index/index.parquet')
assert PARQUET_PATH.exists(), f'Not found: {PARQUET_PATH}'
df = pd.read_parquet(PARQUET_PATH)
print('rows:', len(df))
print('cols:', len(df.columns))
display(df.head(3))


rows: 20015
cols: 24


,ad_id,url,collected_at,price,area,price_per_m2,log_price_per_m2,rooms,district,building_type,...,condition_norm,condition_source,condition_confidence,year_built,floor,floors_total,latitude,longitude,image_paths,image_urls
0,1000113211,https://krisha.kz/a/show/1000113211,2026-04-13 18:04:01.118450+00:00,37000000,43.0,860465.116279,13.665228,2,Бостандыкский р-н,панельный,...,unknown,None,0.0,1971,3,4,43.230280,76.924642,"[data\images\1000113211\01.jpg, data\images\10...",[https://krisha-photos.kcdn.online/webp/ec/ec1...
1,1000116005,https://krisha.kz/a/show/1000116005,2026-04-13 18:04:02.430418+00:00,38500000,58.0,663793.103448,13.405726,3,Бостандыкский р-н,кирпичный,...,unknown,None,0.0,1963,2,4,43.224430,76.889152,"[data\images\1000116005\01.jpg, data\images\10...",[https://krisha-photos.kcdn.online/webp/20/203...
2,1000119398,https://krisha.kz/a/show/1000119398,2026-04-13 18:04:03.884454+00:00,28500000,42.3,673758.865248,13.420628,1,Жетысуский р-н,панельный,...,unknown,None,0.0,1987,1,5,43.324243,76.916423,"[data\images\1000119398\01.jpg, data\images\10...",[https://krisha-photos.kcdn.online/webp/5f/5f5...


In [2]:
required = [
    'ad_id','url','price','area','price_per_m2','log_price_per_m2',
    'rooms','district','building_type','residential_complex','year_built','floor','floors_total',
    'latitude','longitude','image_paths','image_urls',
    'description','object_type','condition_raw','condition_norm','condition_source','condition_confidence','collected_at',
]
missing = [c for c in required if c not in df.columns]
print('missing:', missing)
assert not missing, f'Missing columns: {missing}'


missing: []


In [3]:
df['price_per_m2'] = pd.to_numeric(df['price_per_m2'], errors='coerce')
df['area'] = pd.to_numeric(df['area'], errors='coerce')
df['condition_confidence'] = pd.to_numeric(df['condition_confidence'], errors='coerce')
df['collected_at'] = pd.to_datetime(df['collected_at'], errors='coerce', utc=True)
print('null ratio:')
display((df.isna().mean().sort_values(ascending=False).head(20)).to_frame('null_ratio'))
print('condition_norm:')
display(df['condition_norm'].fillna('unknown').astype(str).value_counts().head(10))
print('object_type:')
display(df['object_type'].fillna('unknown').astype(str).value_counts().head(10))
print('collected_at range:', df['collected_at'].min(), '->', df['collected_at'].max())


null ratio:


,null_ratio
residential_complex,0.435623
condition_raw,0.386410
condition_source,0.256957
description,0.002548
image_paths,0.000000
longitude,0.000000
latitude,0.000000
floors_total,0.000000
floor,0.000000
year_built,0.000000


condition_norm:


condition_norm
fresh      6984
average    5648
unknown    5358
needs      2025
Name: count, dtype: int64

object_type:


object_type
flat    20015
Name: count, dtype: int64

collected_at range: 2026-04-13 18:04:01.118450+00:00 -> 2026-04-14 03:38:02.440860+00:00
